<img src='https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true' align="right" width="100%"/>

<font color="#138D75">**EUMETSAT DestinE User Engagement Service**</font> <br>
**Copyright:** 2026 European Union <br>
**License:** MIT <br>
**Authors:** EUMETSAT DestinE User Engagement Service: Ben Loveday (EUMETSAT/Innoflair UG), Joana Brito (EUMETSAT/Innoflair UG), Madalina Ungur (EUMETSAT/Solenix)

<div class="alert alert-block alert-success">
    <h2>EUMETSAT Conference 2026</h2>
    <p><font color="white"><em>From Data to Insights: working with DestinE Data Lake services</em></font></p>
</div>


<div class="alert alert-block alert-warning">
<b> Prerequisites: </b><ul>
   <li>To search and access DEDL data a <a href="https://platform.destine.eu/"> DestinE user account</a> is needed</li>
   <li>To search and access DT data an <a href="https://platform.destine.eu/support-pages/access-policy/"> upgraded access</a> is needed.</li></ul>
<b> References: </b><ul>
    <li><a href="https://platform.destine.eu/services/documents-and-api/doc/?service_name=climate-dt-user-guide">Climate DT user guide</a></li>
    <li><a href="https://destine-data-lake-docs.data.destination-earth.eu/en/latest/dedl-discovery-and-data-access/Use-of-Harmonized-Data-Access/Use-of-Harmonized-Data-Access.html">DestinE Data Lake (DEDL) Harmonized Data Access (HDA) documentation</a> </li>
    <li> <a href="https://destine.ecmwf.int/climate-change-adaptation-digital-twin-climate-dt/">Climate Change Adaptation DT (Climate DT)</a></li></ul>
<b> Credit: </b><ul>
    <li> Earthkit and Polytope (used from HDA) are both packages provided by the European Centre for Medium-Range Weather Forecasts (ECMWF).</li></ul>
</div>


# From Flood Observation to Climate Context with DestinE STACK: Storm Boris

<div style="margin: 6px 0;">
  <a href="https://jupyter.central.data.destination-earth.eu/user-redirect/lab/tree/DestinE-DataLake-Lab/STACK/DEDL-STACK-DT-CLIMATE.ipynb" target="_blank" style="text-decoration: none;">
    <span class="launch">🚀 Launch in JupyterHub</span>
  </a>
</div>

### Data Used

| Dataset | S3 Location | Purpose | HDA collection ID | HDA collection description |
|:--------------------:|:-----------------------:|:-------------:|:------:|:------:|
| Global Flood Monitoring (GFM) | Central | Satellite-derived flood extent | - | - |
| Climate DT – `hist` | LUMI | Present-day storyline precipitation | EO.ECMWF.DAT.D1.DT_CLIMATE.G2.STORY-NUDGING_HIST_IFS-FESOM.R1 | <a href="https://data.destination-earth.eu/data-portfolio/EO.ECMWF.DAT.D1.DT_CLIMATE.G2.STORY-NUDGING_HIST_IFS-FESOM.R1" target="_blank">Description</a> |
| Climate DT – `cont` | LUMI | Past-climate storyline precipitation | EO.ECMWF.DAT.D1.DT_CLIMATE.G2.STORY-NUDGING_CONT_IFS-FESOM.R1 | <a href="https://data.destination-earth.eu/data-portfolio/EO.ECMWF.DAT.D1.DT_CLIMATE.G2.STORY-NUDGING_CONT_IFS-FESOM.R1" target="_blank">Description</a> |
| Climate DT – `Tplus2.0K` | LUMI | +2°C storyline precipitation | EO.ECMWF.DAT.D1.DT_CLIMATE.G2.STORY-NUDGING_TPLUS2.0K_IFS-FESOM.R1 | <a href="https://data.destination-earth.eu/data-portfolio/EO.ECMWF.DAT.D1.DT_CLIMATE.G2.STORY-NUDGING_TPLUS2.0K_IFS-FESOM.R1" target="_blank">Description</a> |

### Learning outcomes

After running this notebook, you will be able to:

* Set up and authenticate access to the DestinE Data Lake STACK service.
* Create and manage Dask clusters across the DEDL bridges.
* Understand how computations are routed to the location where data are stored.
* Access prepared Zarr datacubes directly from DEDL Islet Storage S3.
* Use Dask to process data close to where they are stored.
* Compare a satellite-derived flood observation with precipitation from three Climate DT storyline experiments.
* Bring only small derived analysis products back to the notebook for interpretation.

### Outline

This notebook demonstrates the **STACK service** of the DestinE Data Lake using the European flooding associated with **Storm Boris in September 2024**. The workflow follows the principle of **data-proximate computing**: instead of moving large datasets to the user or between bridges, computations are sent to the DEDL location where the relevant data are stored.

The notebook starts from **prepared Zarr stores**. The Storm Boris study area is defined once near the start of the notebook and reused for the Climate DT spatial subset and the interactive map extents. Data preparation is deliberately separated from the STACK processing demonstrated here. The GFM flood extent was prepared from the relevant `ENSEMBLE_FLOOD` GeoTIFF tiles and staged at Central. The Climate DT storyline data were retrieved through HDA, spatially subset to the regional domain, retained on their native H128/HEALPix representation, written as Zarr v2, and staged at LUMI. No artificial regridding or downscaling is applied.

The prepared Zarr stores are held in the DEDL **Islet Storage Service (S3)**. In this workflow, we use the S3 capability of Islet Storage as the persistent storage layer, while STACK provides the distributed Dask compute layer. The notebook accesses those stores directly through the Central and LUMI S3 endpoints, while STACK provides the Dask compute layer. This separation of storage and compute is central to the data-proximate workflow.

#### Climate DT storyline experiments

The three Climate DT datasets represent the same observed weather event under different climate states: `hist` represents the present-day storyline, `cont` the past-climate storyline, and `Tplus2.0K` a climate 2°C warmer than pre-industrial conditions. Storyline simulations use nudging to retain the large-scale weather evolution of the observed event while allowing the simulated event to respond to the different climate background.

#### STACK workflow

1. Import the required Python dependencies.
2. Define the helper functions used throughout the notebook.
3. Authenticate with the DestinE Data Lake.
4. Create Dask clusters on the DEDL bridges.
5. Inspect the available Dask dashboards.
6. Connect to Central and LUMI Islet Storage S3.
7. Read and process the GFM flood data at Central.
8. Read the three Climate DT storyline datasets at LUMI.
9. Use the LUMI Dask cluster to calculate regional rainfall time series and cumulative rainfall for each storyline.
10. Bring the small derived results together to compare the observed flood event with the three Climate DT storylines.


<div class="alert alert-info" role="alert">

## Importing dependencies

</div>


We begin by importing the libraries used throughout the notebook. The workflow uses DestinE authentication and STACK cluster management, together with `xarray` and `s3fs` for reading Zarr data from Islet Storage S3. `Folium` provides the interactive maps, while `matplotlib` is used for the derived rainfall comparison.


Import the Python dependencies used throughout the notebook and enable xarray attribute preservation for STACK location routing.

In [ ]:
import destinelab as deauth                           # DestinE Data Lake authentication and access
import json                                           # Read and write JSON data
import os                                             # Interact with the operating system
from getpass import getpass                           # Securely prompt for passwords
from pathlib import Path                              # Construct and manage system file paths
import numpy as np                                    # Work with numerical arrays
import xarray as xr                                   # Work with multidimensional data
import s3fs                                           # Access object storage
import folium                                         # Create interactive maps
from folium.plugins import TimestampedGeoJson         # Animate time-varying geographic data
from folium.raster_layers import ImageOverlay         # Display gridded raster data as a map overlay
import matplotlib.pyplot as plt                       # Plot derived Climate DT time series
import ipywidgets as widgets                          # Create interactive notebook controls
from IPython.display import display, IFrame           # Display dashboards and notebook content
from rich.prompt import Prompt                        # Prompt for usernames
from rich.console import Console                      # Display formatted output
from dedl_stack_client.authn import DaskOIDC          # Authenticate with DEDL STACK
from dedl_stack_client.dask import DaskMultiCluster   # Manage DEDL Dask clusters

import warnings
warnings.filterwarnings("ignore")

xr.set_options(keep_attrs=True);

### Defining the study area

The Storm Boris study area is defined once here and reused throughout the notebook. The same bounding box is used to subset the native Climate DT H128 cells and, where possible, to set the default extent of the interactive maps.

The approximate domain is **8.5–16.5°E, 49–53°N**, covering the main Central European region shown in the case-study visualisation.


In [ ]:
# Storm Boris study area
bbox = {
    "lon_min": 8.5,
    "lon_max": 16.5,
    "lat_min": 49.0,
    "lat_max": 53.0,
}

<div class="alert alert-info" role="alert">

## Defining functions

</div>


The following helper functions are used later in the notebook. They are kept separate from the main workflow so that the processing steps remain easy to follow.

* `show_dashboard` displays the Dask dashboard for the selected DEDL bridge.
* `preprocess_dataset` creates a coarser GFM representation for visualisation only.
* `climate_dt_animation` creates a compact six-hourly animation of one Climate DT storyline using a common rainfall scale.

The function source cells are hidden by default; expand them if you want to inspect the implementation.


Define the dashboard helper used to display the Dask dashboard for the currently selected DEDL bridge.

In [ ]:
def show_dashboard(change=None):
    with dashboard:
        dashboard.clear_output(wait=True)
        cluster = clusters[dropdown.value]
        display(IFrame(
            src=cluster.dashboard_link,
            width="100%",
            height=700,
        ))

Define the GFM visualisation preprocessing helper. The `max` option used below preserves the binary flood mask when coarsening.

In [ ]:
def preprocess_dataset(data_array: xr.DataArray, method: str = "mean"):
    data_array = data_array.squeeze()
    steps = 500 // data_array.attrs["resolution"]
    coarsened = data_array.coarsen({"y": steps, "x": steps}, boundary="trim")

    if method == "median":
        return (coarsened.median() > 0).astype("float32")
    elif method == "mean":
        return coarsened.mean()
    elif method == "max":
        return coarsened.max()
    else:
        raise NotImplementedError(method)

Define the Climate DT animation helper. It creates an animation for one storyline using the common rainfall scale defined later. The map extent is taken from the study-area bounding box.

In [ ]:
def climate_dt_animation(
    da,
    vmax,
    title="Climate DT rainfall",
    cmap=plt.cm.YlOrRd,
):

    rainfall = da * 3600.0
    features = []

    for t, timestamp in enumerate(da.time.values):

        for i, value in enumerate(rainfall.isel(time=t).values):

            if not np.isfinite(value) or value <= 0:
                continue

            fraction = min(float(value) / vmax, 1.0)
            colour = cmap(fraction)

            colour = "#{:02x}{:02x}{:02x}".format(
                int(colour[0] * 255),
                int(colour[1] * 255),
                int(colour[2] * 255),
            )

            features.append({
                "type": "Feature",
                "geometry": {
                    "type": "Point",
                    "coordinates": [
                        float(da.longitude.values[i]),
                        float(da.latitude.values[i]),
                    ],
                },
                "properties": {
                    "times": [
                        np.datetime_as_string(timestamp, unit="s")
                    ],
                    "icon": "circle",
                    "iconstyle": {
                        "fillColor": colour,
                        "fillOpacity": 1.0,
                        "stroke": False,
                        "radius": 6,
                    },
                },
            })

    # Determine animation timestep from the actual data
    if da.sizes["time"] > 1:
        timestep = da.time.values[1] - da.time.values[0]
        timestep_hours = int(
            timestep / np.timedelta64(1, "h")
        )
    else:
        timestep_hours = 1

    period = f"PT{timestep_hours}H"
    duration = period

    clim_map = folium.Map(
        location=[
            (bbox["lat_min"] + bbox["lat_max"]) / 2,
            (bbox["lon_min"] + bbox["lon_max"]) / 2,
        ],
        zoom_start=6,
        tiles=None,
        control_scale=True,
    )

    folium.TileLayer(
        tiles="OpenStreetMap",
        opacity=0.5,
        attr="© OpenStreetMap contributors",
    ).add_to(clim_map)

    clim_map.fit_bounds([
        [bbox["lat_min"], bbox["lon_min"]],
        [bbox["lat_max"], bbox["lon_max"]],
    ])

    TimestampedGeoJson(
        {
            "type": "FeatureCollection",
            "features": features,
        },
        period=period,
        duration=duration,
        add_last_point=False,
        auto_play=False,
        loop=True,
        loop_button=True,
        date_options="YYYY-MM-DD HH:mm",
        time_slider_drag_update=True,
    ).add_to(clim_map)

    cmap_colours = [
        "#{:02x}{:02x}{:02x}".format(
            int(r * 255),
            int(g * 255),
            int(b * 255),
        )
        for r, g, b, _ in cmap(np.linspace(0, 1, 10))
    ]

    colormap = folium.LinearColormap(
        colors=cmap_colours,
        vmin=0,
        vmax=vmax,
        caption="Rainfall (mm/hour)",
    )

    colormap.add_to(clim_map)

    # Add title in the top-right, below the map controls
    title_html = f"""
    <div style="
        position: fixed;
        top: 70px;
        right: 10px;
        z-index: 9999;
        background-color: rgba(255, 255, 255, 0.9);
        padding: 6px 12px;
        border: 1px solid #999;
        border-radius: 4px;
        font-size: 16px;
        font-weight: bold;
        pointer-events: none;
    ">
        {title}
    </div>
    """

    clim_map.get_root().html.add_child(
        folium.Element(title_html)
    )

    return clim_map

<div class="alert alert-info" role="alert">

## Authenticating the DEDL

</div>


DEDL services require authentication. As in the HDA notebooks, credentials are read from the local `.dedl/credentials` file when available. If the file is not present, the notebook prompts for the required credentials.

The credentials are used only for authentication; they are not included in the data-processing workflow. **Do not commit the credentials file to source control or share it with other users.**


Read the local DEDL credentials, prompting for them only when no credentials file is available.

In [ ]:
credentials_file = Path(Path.home() / '.dedl' / 'credentials')

if os.path.exists(credentials_file):
    with open(credentials_file, "r") as f:
        config = json.load(f)
    DESP_USERNAME = config["username"]
    DESP_PASSWORD = config["desp_password"]
else:
    # creating authentication file
    DESP_USERNAME = input("Please input your DESP username or email: ")
    DESP_PASSWORD = getpass("Please input your DESP password: ")
    OIDC_PASSWORD = getpass("Please input your OIDC password (if known): ")

    config = {"username": DESP_USERNAME,
              "desp_password": DESP_PASSWORD,
              "oidc_password": OIDC_PASSWORD}
    try:
        os.makedirs(os.path.dirname(credentials_file), exist_ok=True)        
        with open(credentials_file, "w") as f:
            json.dump(config, f, indent=4)
    except:
        pass

<div class="alert alert-info" role="alert">

## DestinE Data Lake (DEDL) Stack Client

</div>


The [DEDL Stack Client](https://github.com/destination-earth/DestinE_EUMETSAT_DEDL_Stack_Client) provides the interface used here to manage Dask clusters across DEDL bridges.

The important concept is the `location` attribute attached to each data object. STACK uses that information to route a computation to the bridge where the data are stored, allowing the notebook to work with Central and LUMI data without moving the source datasets between them.


Authenticate the STACK client using the DEDL OIDC credentials.

In [ ]:
myAuth = DaskOIDC(username=DESP_USERNAME)
print("DEDL OIDC authentication successful")

Create the multi-location Dask manager and start the available DEDL bridge clusters.

In [ ]:
myDEDLClusters = DaskMultiCluster(auth=myAuth)
myDEDLClusters.new_cluster()

Display the dashboard links returned by the DEDL cluster manager.

In [ ]:
print("----Cluster links----")
myDEDLClusters.get_cluster_url()

<div class="alert alert-info" role="alert">

## Dask dashboards

</div>


Each DEDL bridge has its own Dask cluster and dashboard. The selector below is only for inspecting the available cluster dashboards; it does not move data or affect computation routing.


Create the bridge selector and display the Dask dashboard for the selected location.

In [ ]:
clusters = myDEDLClusters.cluster

dropdown = widgets.Dropdown(
    options=list(clusters.keys()),
    value="central",
    description="Bridge:",
    style={"description_width": "initial"},
)

dashboard = widgets.Output()
dropdown.observe(show_dashboard, names="value")

display(dropdown, dashboard)
show_dashboard()

<div class="alert alert-info" role="alert">

## Accessing data from DEDL object storage

</div>


The prepared GFM and Climate DT datasets are stored as Zarr datacubes in the **DEDL Islet Storage Service (S3)**.

For this demonstration:

- **Central** hosts the prepared GFM observation.
- **LUMI** hosts the three prepared Climate DT storyline datasets.

The S3 object stores provide persistent data access, while STACK supplies the Dask compute layer. The combination allows computation to be performed close to the data rather than transferring the full datacubes to the notebook or between bridges.


Create S3 filesystem connections to the Central and LUMI Islet Storage endpoints used by the prepared Zarr stores.

In [ ]:
s3fs_central = s3fs.S3FileSystem(
    anon=True,
    use_ssl=True,
    client_kwargs={"endpoint_url": "https://s3.central.data.destination-earth.eu"},
)

s3fs_lumi = s3fs.S3FileSystem(
    anon=True,
    use_ssl=True,
    client_kwargs={"endpoint_url": "https://s3.lumi.data.destination-earth.eu"},
)

List the training objects available in the Central S3 bucket so that the prepared GFM store can be confirmed.

In [ ]:
print("Central:")
s3fs_central.ls("dues-training-central")

List the training objects available in the LUMI S3 bucket so that the three prepared Climate DT stores can be confirmed.

In [ ]:
print("\nLUMI:")
s3fs_lumi.ls("dues-training-lumi")

The GFM flood extent for **18 September 2024** is read from the prepared Zarr store in Central Islet Storage. This is the satellite-observation side of the demonstration.


Open the prepared GFM Zarr store from Central S3 as a Dask-backed DataArray and tag it with its compute location.

In [ ]:
flood_map = xr.open_zarr(
    store=s3fs.S3Map(
        root="dues-training-central/boris_gfm_20240918.zarr",
        s3=s3fs_central,
        check=False,
    ),
    decode_coords="all",
)["flood"].assign_attrs(
    location="central",
    resolution=20,
)

flood_map

## Processing close to the data

The flood map is a Dask-backed `xarray.DataArray`; opening the Zarr store does not load the full array into the notebook.

The flooded-area calculation is submitted through STACK. Because `flood_map` carries `location="central"`, the calculation is routed to the **Central** Dask cluster, where the GFM store resides.

This is the core data-proximate pattern: **the computation moves to the data rather than the data moving to the computation**.


Build the lazy flooded-area calculation from the 20 m binary GFM mask. The result is expressed in square kilometres.

In [ ]:
# 20 m pixels: convert flooded pixel count to km²
flooded_area_ = flood_map.sum() * 20 * 20 / 1_000_000


Submit the flooded-area calculation through STACK and return the single derived value from the Central cluster.

*Note: this is the point at which Central does some work, so we can see the task on the central dashboard*

In [ ]:
flooded_area = myDEDLClusters.compute(flooded_area_, sync=True)
console = Console()
console.print(f"Flooded area: {flooded_area.data} km²")

The native GFM mask is 20 m resolution, which is unnecessarily detailed for the interactive overview map. This preprocessing creates a coarser visual representation using the maximum value in each block. It changes only the visualisation product; the flooded-area calculation above uses the original 20 m data.


Prepare the coarser GFM representation for visualisation. This remains lazy until submitted to the Central cluster.

In [ ]:
flood_prep_ = preprocess_dataset(flood_map, "max")

Submit the GFM visualisation preprocessing to Central and materialise only the reduced array needed for the map.

In [ ]:
flood_prep = myDEDLClusters.compute(flood_prep_, sync=True)
flood_prep

### Visualising the observed flood extent

The processed GFM mask is displayed as an interactive Folium overlay. The map uses the Storm Boris study-area bounding box defined near the start of the notebook as its default extent, providing the observational reference for the Climate DT analysis.

Render the processed GFM flood extent as an interactive Folium map.

In [ ]:
# Display the GFM flood extent using its own spatial extent

map1 = folium.Map(
    location=[
        float(flood_prep.y.mean()),
        float(flood_prep.x.mean()),
    ],
    zoom_start=8,
    tiles="OpenStreetMap",
    control_scale=True,
)

flood_array = flood_prep.values

rgba = np.zeros((*flood_array.shape, 4), dtype=np.uint8)

flood_mask = flood_array > 0

rgba[flood_mask, 0] = 220
rgba[flood_mask, 1] = 40
rgba[flood_mask, 2] = 40
rgba[flood_mask, 3] = 180

bounds = [
    [float(flood_prep.y.min()), float(flood_prep.x.min())],
    [float(flood_prep.y.max()), float(flood_prep.x.max())],
]

ImageOverlay(
    image=rgba,
    bounds=bounds,
    opacity=1.0,
    name="GFM flood extent",
).add_to(map1)

folium.LayerControl().add_to(map1)

map1

<div class="alert alert-info" role="alert">

## Processing Climate DT data at LUMI

</div>

The three prepared Climate DT storyline datasets remain stored at the **LUMI** bridge. We open the Zarr stores directly from LUMI Islet Storage S3 and immediately apply the study-area subset defined near the start of the notebook.

The data remain on their native H128 representation: we select only cells whose centres fall inside the study-area bounding box and do not regrid or downscale the Climate DT data. The small boolean spatial mask is evaluated locally, while the rainfall data remain Dask-backed at LUMI.

Each resulting DataArray is tagged with `location="lumi"` so that subsequent STACK computations are routed to the LUMI Dask cluster.

Open the three prepared Climate DT storyline Zarr stores from LUMI S3, apply the study-area subset, and tag each resulting DataArray for LUMI compute routing.

In [ ]:
climatedt = {}

for experiment in ["hist", "cont", "Tplus2.0K"]:

    path = (f"s3://dues-training-lumi/climatedt/"f"{experiment}_20240910_20240920.zarr")

    ds = xr.open_zarr(
        path,
        storage_options={
            "anon": True,
            "client_kwargs": {
                "endpoint_url": "https://s3.lumi.data.destination-earth.eu"
            },
        },
    )

    cell_mask = ((ds.longitude >= bbox["lon_min"]) & (ds.longitude <= bbox["lon_max"]) 
                 & (ds.latitude >= bbox["lat_min"]) & (ds.latitude <= bbox["lat_max"])).compute()

    climatedt[experiment] = (ds["avg_tprate"].isel(cell=cell_mask).assign_attrs(location="lumi"))

    print(
        f"{experiment}: "
        f"{climatedt[experiment].sizes['time']} timesteps × "
        f"{climatedt[experiment].sizes['cell']} cells"
    )

### Regional rainfall time series

`avg_tprate` is a precipitation rate in `kg m-2 s-1`. For liquid-water equivalent precipitation this is numerically equivalent to `mm s-1`, so multiplying by 3600 converts it to **mm/hour**.

For each storyline we calculate the spatial mean across the selected native H128 cells for every hourly timestep. The same regional domain is used for all three experiments, so the resulting time series can be compared directly.

The mean time series is calculated at LUMI; only the small derived result is returned to the notebook.

Calculate the regional mean rainfall rate for each storyline at LUMI, then return only the small derived time series.

*Note: this is the point at which Lumi does some work, so we can see the task on the lumi dashboard*

In [ ]:
rainfall_timeseries_ = {}

for experiment, da in climatedt.items():
    rainfall_mm_h = da * 3600.0
    rainfall_timeseries_[experiment] = rainfall_mm_h.mean(dim="cell")

rainfall_timeseries = {}

for experiment, da in rainfall_timeseries_.items():
    rainfall_timeseries[experiment] = myDEDLClusters.compute(
        da,
        sync=True,
    )

    print(
        f"{experiment}: "
        f"{rainfall_timeseries[experiment].sizes['time']} hourly values"
    )

### Cumulative rainfall

The hourly regional rainfall rates can also be accumulated through time to show the total rainfall received by the selected region during the event period. This cumulative calculation is performed at LUMI, and only the resulting small time series is returned.

Calculate cumulative regional rainfall for each storyline at LUMI and return the resulting time series.

In [ ]:
cumulative_rainfall_ = {
    experiment: da.cumsum(dim="time")
    for experiment, da in rainfall_timeseries_.items()
}

cumulative_rainfall = {}

for experiment, da in cumulative_rainfall_.items():
    cumulative_rainfall[experiment] = myDEDLClusters.compute(
        da,
        sync=True,
    )

### Comparing the three Climate DT storylines

The three derived rainfall time series now provide a common comparison of the Storm Boris period. Because the experiments represent the same storyline event under different climate states, differences between the curves illustrate how the simulated precipitation associated with that event changes between the past-climate, present-day and +2°C conditions.

This is a **what-if storyline comparison**, not a weather forecast and not an estimate of how the probability of the event changes.

Plot the three derived regional rainfall time series using consistent labels and units.

In [ ]:
labels = {
    "hist": "HIST – present day",
    "cont": "CONT – past climate",
    "Tplus2.0K": "T+2.0K – +2°C",
}

fig, ax = plt.subplots(figsize=(11, 5))

for experiment, da in rainfall_timeseries.items():
    ax.plot(
        da.time.values,
        da.values,
        label=labels[experiment],
    )

ax.set_xlabel("Time")
ax.set_ylabel("Regional mean rainfall (mm/hour)")
ax.set_title("Climate DT storyline precipitation – Storm Boris period")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Create a small six-hourly subset of each storyline for interactive visualisation. Only this subset is materialised from LUMI; the full prepared datasets remain available for the statistics above.


In [ ]:
# Use the first 72 hours at six-hourly intervals for the interactive visualisation.
animation_data = {
    experiment: myDEDLClusters.compute(
        climatedt[experiment].isel(time=slice(0, -1, 3)),
        sync=True,
    )
    for experiment in ["hist", "cont", "Tplus2.0K"]
}

for experiment, da in animation_data.items():
    print(
        f"{experiment}: "
        f"{da.sizes['time']} timesteps × "
        f"{da.sizes['cell']} cells"
    )

Calculate one common maximum rainfall value across the full **spatially subsetted** period and all three storylines. This keeps the colour scale identical when switching between storyline animations.


In [ ]:
# Use one common maximum across all three full-period datasets.
vmax_ = {}

for experiment, da in climatedt.items():
    vmax_[experiment] = myDEDLClusters.compute(
        (da * 3600.0).max(),
        sync=True,
    )

vmax = max(float(value.data) for value in vmax_.values())

print(f"Common rainfall scale: 0–{vmax:.1f} mm/hour")

Select which Climate DT storyline to display in the animation. The same study-area extent, six-hourly timestep and common rainfall scale are used for every storyline.


In [ ]:
story_line = "Tplus2.0K" # "hist" or "cont"

Render the selected Climate DT storyline as an interactive six-hourly rainfall animation using the common rainfall scale.


In [ ]:
map2 = climate_dt_animation(
    animation_data[story_line],
    5,
    f"ClimateDT rainfall rate around Storm Boris for the {story_line} story line",
)

map2

<div class="alert alert-info" role="alert">

## What has happened in the STACK workflow?

</div>

We have now demonstrated the two sides of a data-proximate workflow:

- **Central Dask:** processed the satellite-derived GFM flood observation where it is stored.
- **LUMI Dask:** processed the three Climate DT storyline datasets where they are stored.
- **Notebook:** received only small derived results needed for comparison and visualisation.

The important result is not simply the rainfall plot. It is the architecture: **different datasets can remain at their respective DEDL locations while computation is sent to them, and only the information needed for the next analysis step is brought back together.**

For the application story, the GFM observation provides the satellite evidence of flooding, while the Climate DT storylines provide a climate-contextualised way to replay the same extreme event under different climate states. The storyline results should therefore be interpreted as physically consistent *what-if* experiments rather than as a flood forecast.


<div class="alert alert-info" role="alert">

## Conclusion and shutdown

</div>

This notebook has taken a complete data-proximate path from observation to climate context: a satellite-derived flood extent was processed at Central, while three Climate DT storyline precipitation datasets were processed at LUMI. The notebook then brought together only the resulting flood-area metric and compact rainfall time series.

The key STACK lesson is that **analysis does not require the source datasets to be brought to one place**. Storage remains close to the relevant infrastructure, computation is routed to that storage location, and only the derived information required for interpretation needs to move.

When you have finished exploring the results, shut down the DEDL Dask clusters to release the allocated resources.


Shut down the DEDL Dask clusters and release the allocated compute resources.

In [ ]:
myDEDLClusters.shutdown()

<div class="alert alert-info" role="alert">

## References and further reading

</div>

### DestinE Data Lake

- <a href="https://destine-data-lake-docs.data.destination-earth.eu/en/latest/dedl-big-data-processing-services/Stack-service/Stack-service.html" target="_blank">DEDL STACK service documentation</a>
- <a href="https://destination-earth.github.io/DestinE-DataLake-Gallery/stack-python-client-dask/" target="_blank">STACK Python Client / Dask example</a>
- <a href="https://destination-earth.github.io/DestinE-DataLake-Gallery/stack/" target="_blank">STACK notebook gallery</a>
- <a href="https://destine-data-lake-docs.data.destination-earth.eu/en/latest/dedl-big-data-processing-services/Islet-service/s3/s3.html" target="_blank">DEDL Islet Storage Service – S3 documentation</a>
- <a href="https://destine-data-lake-docs.data.destination-earth.eu/en/latest/dedl-discovery-and-data-access/Use-of-Harmonized-Data-Access/Use-of-Harmonized-Data-Access.html" target="_blank">DEDL Harmonised Data Access (HDA) documentation</a>

### Climate DT and storyline simulations

- <a href="https://destine.ecmwf.int/climate-digital-twin/" target="_blank">ECMWF – Climate Change Adaptation Digital Twin</a>
- <a href="https://destine.ecmwf.int/news/replaying-extreme-weather-how-storyline-simulations-help-us-prepare-for-climate-change/" target="_blank">ECMWF – Replaying extreme weather with storyline simulations</a>
- <a href="https://destine-data-lake-docs.data.destination-earth.eu/en/latest/_downloads/4d64fc9f416b68551a102902bdca1cf3/DestinE-System-Framework-Data-Portfolio.pdf" target="_blank">DestinE System Framework – Data Portfolio</a>

### Source repositories

- <a href="https://github.com/destination-earth/DestinE-DataLake-Lab" target="_blank">DestinE Data Lake Lab</a>
- <a href="https://github.com/destination-earth/DestinE_EUMETSAT_DEDL_Stack_Client" target="_blank">DEDL STACK Python client</a>


<img src='https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true' align="right" width="100%"/>